In [1]:
!pip install numpy pandas scikit-learn tensorflow


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model
from datetime import datetime
import os

print("="*70)
print("📈 ADBL STOCK PRICE DAILY PREDICTOR")
print("="*70)
print(f"🕐 Date & Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Step 1: Load data
print("\n[1/5] Loading stock data...")
data = pd.read_csv("./STOCKNEPSE/NEPSEDATA/adbl.csv")
data = data[['Close']]
data.dropna(inplace=True)
data = data.iloc[::-1].reset_index(drop=True)
data['Close'] = data['Close'].str.replace(',', '').astype(float)

print(f"✅ Total historical data points: {len(data)}")

# Step 2: Scale data
print("\n[2/5] Scaling data...")
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data.values.reshape(-1, 1))
print("✅ Data scaled successfully")

# Step 3: Prepare last 60 days
print("\n[3/5] Preparing last 60 days of data...")
last_60_days = scaled_data[-60:].reshape(1, 60, 1)
print("✅ Input data prepared")

# Step 4: Load trained model
print("\n[4/5] Loading trained model...")
if os.path.exists("best_lstm_stock_model.h5"):
    model = load_model("best_lstm_stock_model.h5")
    print("✅ Model loaded successfully!")
else:
    print("❌ ERROR: Model file 'best_lstm_stock_model.h5' not found!")
    print("   Please run the training script first.")
    exit()

# Step 5: Make prediction
print("\n[5/5] Making prediction...")
predicted_scaled = model.predict(last_60_days, verbose=0)
predicted_price = scaler.inverse_transform(predicted_scaled)[0][0]
print("✅ Prediction complete!")

# Calculate changes
current_price = data['Close'].iloc[-1]
change = predicted_price - current_price
change_percent = (change / current_price) * 100

# Display results
print("\n" + "="*70)
print("📊 PREDICTION RESULTS")
print("="*70)
print(f"📅 Today's Date:           {datetime.now().strftime('%Y-%m-%d')}")
print(f"💰 Current Price (Today):  Rs. {current_price:,.2f}")
print(f"🔮 Predicted (Tomorrow):   Rs. {predicted_price:,.2f}")
print(f"📈 Expected Change:        Rs. {change:+,.2f} ({change_percent:+.2f}%)")
print("="*70)

# Prediction interpretation
if abs(change_percent) < 0.5:
    emoji = "⚪"
    status = "STABLE"
    message = f"Minimal change expected (less than 0.5%)"
elif change > 0:
    if change_percent > 2:
        emoji = "🟢🟢"
        status = "STRONG UP"
        message = f"Strong upward movement predicted (+{change_percent:.2f}%)"
    else:
        emoji = "🟢"
        status = "UP"
        message = f"Moderate increase predicted (+{change_percent:.2f}%)"
else:
    if change_percent < -2:
        emoji = "🔴🔴"
        status = "STRONG DOWN"
        message = f"Strong downward movement predicted ({change_percent:.2f}%)"
    else:
        emoji = "🔴"
        status = "DOWN"
        message = f"Moderate decrease predicted ({change_percent:.2f}%)"

print(f"\n{emoji} {status}")
print(f"💬 {message}")

# Model accuracy reminder
print("\n" + "="*70)
print("⚠️  MODEL ACCURACY INFORMATION")
print("="*70)
print("✅ Price Accuracy:      97.03% (R² Score)")
print("✅ Average Error:       1.78% (MAPE)")
print("✅ Typical Error:       Rs. 5.44 (MAE)")
print("⚠️  Direction Accuracy: 52.51% (Barely better than random)")
print("="*70)
print("📌 USE AS REFERENCE ONLY - NOT FINANCIAL ADVICE")
print("="*70)

# Show recent history
print("\n📈 Recent 10-Day Price History:")
print("-"*70)
recent = data[['Close']].tail(10).copy()
recent['Close'] = recent['Close'].apply(lambda x: f"Rs. {x:,.2f}")
recent.index = range(len(recent), 0, -1)
recent.columns = ['Price']
print(recent.to_string())

# Calculate recent trend
last_5_prices = data['Close'].tail(5).values
trend_change = last_5_prices[-1] - last_5_prices[0]
trend_percent = (trend_change / last_5_prices[0]) * 100

print("\n📊 5-Day Trend Analysis:")
if trend_change > 0:
    print(f"   🟢 Upward trend: +Rs. {trend_change:.2f} (+{trend_percent:.2f}%)")
else:
    print(f"   🔴 Downward trend: Rs. {trend_change:.2f} ({trend_percent:.2f}%)")

print("\n" + "="*70)
print("✅ PREDICTION COMPLETE!")
print("="*70)

📈 ADBL STOCK PRICE DAILY PREDICTOR
🕐 Date & Time: 2026-02-05 20:08:27

[1/5] Loading stock data...
✅ Total historical data points: 3513

[2/5] Scaling data...
✅ Data scaled successfully

[3/5] Preparing last 60 days of data...
✅ Input data prepared

[4/5] Loading trained model...
✅ Model loaded successfully!

[5/5] Making prediction...
✅ Prediction complete!

📊 PREDICTION RESULTS
📅 Today's Date:           2026-02-05
💰 Current Price (Today):  Rs. 295.10
🔮 Predicted (Tomorrow):   Rs. 297.95
📈 Expected Change:        Rs. +2.85 (+0.97%)

🟢 UP
💬 Moderate increase predicted (+0.97%)

⚠️  MODEL ACCURACY INFORMATION
✅ Price Accuracy:      97.03% (R² Score)
✅ Average Error:       1.78% (MAPE)
✅ Typical Error:       Rs. 5.44 (MAE)
⚠️  Direction Accuracy: 52.51% (Barely better than random)
📌 USE AS REFERENCE ONLY - NOT FINANCIAL ADVICE

📈 Recent 10-Day Price History:
----------------------------------------------------------------------
         Price
10  Rs. 304.00
9   Rs. 307.50
8   Rs. 307.00
